# ITO5202 Assessment 1: Analysing Historical Data with System Performance

**Student ID:** 27602966  
**Unit:** ITO5202 Data Processing for Big Data  
**Teaching Period:** TP5  
**Year:** 2026

## Execution Environment

The analysis was executed in the Monash ITO5202 Docker environment using Apache Spark in local mode. Spark was configured with `local[2]`, giving a default parallelism of 2, with 2 GB driver memory. No separate executor memory setting was configured because Spark was run in local mode. The Spark session timezone was set to `America/New_York` before deriving time based attributes from the taxi trip timestamps.

In [1]:
import os
import sys
import pyspark
import time
import statistics

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 2g pyspark-shell"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 1")
    .master("local[2]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/New_York"
)

print("Spark version:", spark.version)
print("Master:", sc.master)
print("Default parallelism:", sc.defaultParallelism)
print("Driver memory:", sc.getConf().get("spark.driver.memory"))
print("Spark UI:", sc.uiWebUrl)

Spark version: 4.1.1
Master: local[2]
Default parallelism: 2
Driver memory: 2g
Spark UI: http://b60ecd721b18:4040


# Part A: Analytical Query Design and Implementation

## 1. Business Query

### 1.1 Business Question

Within each pickup borough, day of the week and hour of the day, which three high volume pickup and drop off routes have the greatest travel speed deficit compared with other trips operating in the same borough and time period?

The analysis also examines whether these routes generate lower metered fare per occupied minute than the corresponding peer benchmark. A route is considered high volume when it contains at least 100 valid trips within the relevant borough, day of week and hourly period.

### 1.2 Justification for Distributed Processing

The dataset contains approximately 38.3 million taxi trip records across 12 monthly Parquet files. The analysis requires explicit schema handling, filtering, two geographical joins, derived time and duration attributes, separate route level and borough level aggregations, a join between the aggregated results and window ranking within each borough and time period. The complete analytical query therefore requires multiple transformation, join, aggregation, shuffle and window processing stages rather than a single aggregation.

## 2. DataFrame Implementation

### 2.1 Schema Definition and Source Inspection

In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampType,
    IntegerType
)

trip_schema_jan = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [4]:
zone_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

In [5]:
zone_df = (
    spark.read
    .option("header", True)
    .schema(zone_schema)
    .csv("data/taxi_zone_lookup.csv")
)

zone_df.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


In [6]:
zone_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [7]:
jan_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-01.parquet"
)

jan_check_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

jan_source_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [9]:
jan_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [10]:
from pyspark.sql.functions import round as spark_round
from pyspark.sql.functions import col

jan_standardised_df = jan_df.select(
    col("VendorID").cast("long").alias("VendorID"),
    col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
    col("passenger_count").cast("long").alias("passenger_count"),
    col("trip_distance").cast("double").alias("trip_distance"),
    col("RatecodeID").cast("long").alias("RatecodeID"),
    col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
    col("PULocationID").cast("long").alias("PULocationID"),
    col("DOLocationID").cast("long").alias("DOLocationID"),
    col("payment_type").cast("long").alias("payment_type"),
    col("fare_amount").cast("double").alias("fare_amount"),
    col("extra").cast("double").alias("extra"),
    col("mta_tax").cast("double").alias("mta_tax"),
    col("tip_amount").cast("double").alias("tip_amount"),
    col("tolls_amount").cast("double").alias("tolls_amount"),
    col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
    col("total_amount").cast("double").alias("total_amount"),
    col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
    col("airport_fee").cast("double").alias("airport_fee")
)

In [11]:
jan_standardised_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [12]:
jan_standardised_df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "airport_fee"
).show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |0.97         |161         |141         |9.3        |0.0        |
|2       |2023-01-01 00:55:08 |2023-01-01 01:01:27  |1              |1.1          |43          |237         |7.9        |0.0        |
|2       |2023-01-01 00:25:04 |2023-01-01 00:37:49  |1              |2.51         |48          |238         |14.9       |0.0        |
|1       |2023-01-01 00:03:48 |2023-01-01 00:13:25  |0              |1.9          |138         |7           |12.1       |1.25       |
|2       |2023-01-01 00:10:29 |2023-01-01 00:21:19  |1        

In [13]:
feb_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-02.parquet"
)

feb_check_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [14]:
for month in range(1, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    print(f"\nMonth: {month:02d}")
    spark.read.parquet(path).printSchema()


Month: 01
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)


Month: 02
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-

In [15]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

later_source_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

In [16]:
from pyspark.sql.functions import col

def standardise_trip_df(df, airport_column):
    return df.select(
        col("VendorID").cast("long").alias("VendorID"),
        col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
        col("passenger_count").cast("long").alias("passenger_count"),
        col("trip_distance").cast("double").alias("trip_distance"),
        col("RatecodeID").cast("long").alias("RatecodeID"),
        col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        col("PULocationID").cast("long").alias("PULocationID"),
        col("DOLocationID").cast("long").alias("DOLocationID"),
        col("payment_type").cast("long").alias("payment_type"),
        col("fare_amount").cast("double").alias("fare_amount"),
        col("extra").cast("double").alias("extra"),
        col("mta_tax").cast("double").alias("mta_tax"),
        col("tip_amount").cast("double").alias("tip_amount"),
        col("tolls_amount").cast("double").alias("tolls_amount"),
        col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        col("total_amount").cast("double").alias("total_amount"),
        col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
        col(airport_column).cast("double").alias("airport_fee")
    )

### 2.2 Monthly Data Loading and Standardisation

In [17]:
jan_raw_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df = standardise_trip_df(
    jan_raw_df,
    "airport_fee"
)

In [18]:
monthly_dfs = [jan_df]

for month in range(2, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    month_raw_df = (
        spark.read
        .schema(later_source_schema)
        .parquet(path)
    )

    month_df = standardise_trip_df(
        month_raw_df,
        "Airport_fee"
    )

    monthly_dfs.append(month_df)

In [19]:
from functools import reduce

trips_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    monthly_dfs
)

In [20]:
trips_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [21]:
trips_df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "airport_fee"
).show(5, truncate=False)

+--------+--------------------+---------------------+---------------+----------+------------+------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|RatecodeID|PULocationID|DOLocationID|airport_fee|
+--------+--------------------+---------------------+---------------+----------+------------+------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |1         |161         |141         |0.0        |
|2       |2023-01-01 00:55:08 |2023-01-01 01:01:27  |1              |1         |43          |237         |0.0        |
|2       |2023-01-01 00:25:04 |2023-01-01 00:37:49  |1              |1         |48          |238         |0.0        |
|1       |2023-01-01 00:03:48 |2023-01-01 00:13:25  |0              |1         |138         |7           |1.25       |
|2       |2023-01-01 00:10:29 |2023-01-01 00:21:19  |1              |1         |107         |79          |0.0        |
+--------+--------------------+-----------------

In [22]:
print("Number of monthly DataFrames:", len(monthly_dfs))
print("Total trip records:", trips_df.count())

Number of monthly DataFrames: 12
Total trip records: 38310226


### 2.3 Derived Analytical Fields

In [23]:
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [24]:
from pyspark.sql.functions import (
    col,
    to_date,
    hour,
    dayofweek,
    unix_timestamp
)

trips_derived_df = (
    trips_df
    .withColumn(
        "pickup_date",
        to_date(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "pickup_hour",
        hour(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "day_of_week",
        dayofweek(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "trip_duration_minutes",
        (
            unix_timestamp(col("tpep_dropoff_datetime"))
            - unix_timestamp(col("tpep_pickup_datetime"))
        ) / 60.0
    )
    .withColumn(
        "pickup_epoch_seconds",
        unix_timestamp(col("tpep_pickup_datetime")).cast("long")
    )
)

In [25]:
trips_derived_df.select(
    "tpep_pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "day_of_week",
    spark_round("trip_duration_minutes", 2).alias("duration_min"),
    "pickup_epoch_seconds"
).show(10, truncate=False)

+--------------------+-----------+-----------+-----------+------------+--------------------+
|tpep_pickup_datetime|pickup_date|pickup_hour|day_of_week|duration_min|pickup_epoch_seconds|
+--------------------+-----------+-----------+-----------+------------+--------------------+
|2023-01-01 00:32:10 |2023-01-01 |0          |1          |8.43        |1672551130          |
|2023-01-01 00:55:08 |2023-01-01 |0          |1          |6.32        |1672552508          |
|2023-01-01 00:25:04 |2023-01-01 |0          |1          |12.75       |1672550704          |
|2023-01-01 00:03:48 |2023-01-01 |0          |1          |9.62        |1672549428          |
|2023-01-01 00:10:29 |2023-01-01 |0          |1          |10.83       |1672549829          |
|2023-01-01 00:50:34 |2023-01-01 |0          |1          |12.3        |1672552234          |
|2023-01-01 00:09:22 |2023-01-01 |0          |1          |10.45       |1672549762          |
|2023-01-01 00:27:12 |2023-01-01 |0          |1          |22.73       

### 2.4 Taxi Zone Lookup Joins

In [26]:
print("Taxi zone records:", zone_df.count())
zone_df.show(5, truncate=False)

Taxi zone records: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [27]:
pickup_zone_df = zone_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone"),
    col("service_zone").alias("pickup_service_zone")
)

dropoff_zone_df = zone_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone"),
    col("service_zone").alias("dropoff_service_zone")
)

In [28]:
trips_joined_df = (
    trips_derived_df
    .join(
        pickup_zone_df,
        on="PULocationID",
        how="left"
    )
    .join(
        dropoff_zone_df,
        on="DOLocationID",
        how="left"
    )
)

In [29]:
trips_joined_df.select(
    col("PULocationID").alias("PU_ID"),
    col("pickup_borough").alias("PU_Borough"),
    col("pickup_zone").alias("PU_Zone"),
    col("DOLocationID").alias("DO_ID"),
    col("dropoff_borough").alias("DO_Borough"),
    col("dropoff_zone").alias("DO_Zone")
).show(5, truncate=25)

+-----+----------+-----------------+-----+----------+---------------------+
|PU_ID|PU_Borough|          PU_Zone|DO_ID|DO_Borough|              DO_Zone|
+-----+----------+-----------------+-----+----------+---------------------+
|  161| Manhattan|   Midtown Center|  141| Manhattan|      Lenox Hill West|
|   43| Manhattan|     Central Park|  237| Manhattan|Upper East Side South|
|   48| Manhattan|     Clinton East|  238| Manhattan|Upper West Side North|
|  138|    Queens|LaGuardia Airport|    7|    Queens|              Astoria|
|  107| Manhattan|         Gramercy|   79| Manhattan|         East Village|
+-----+----------+-----------------+-----+----------+---------------------+
only showing top 5 rows


In [30]:
print("Records before joins:", trips_derived_df.count())
print("Records after joins:", trips_joined_df.count())

Records before joins: 38310226
Records after joins: 38310226


In [31]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum(
        col("pickup_borough").isNull().cast("int")
    ).alias("unmatched_pickup_locations"),

    spark_sum(
        col("dropoff_borough").isNull().cast("int")
    ).alias("unmatched_dropoff_locations")
).show()

+--------------------------+---------------------------+
|unmatched_pickup_locations|unmatched_dropoff_locations|
+--------------------------+---------------------------+
|                         0|                          0|
+--------------------------+---------------------------+



In [32]:
trips_joined_df.filter(
    col("pickup_borough").isNull()
).groupBy(
    "PULocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|PULocationID|count|
+------------+-----+
+------------+-----+



In [33]:
trips_joined_df.filter(
    col("dropoff_borough").isNull()
).groupBy(
    "DOLocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|DOLocationID|count|
+------------+-----+
+------------+-----+



### 2.5 Data Quality Checks and Filtering

In [34]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum((col("trip_distance") <= 0).cast("int")).alias("non_positive_distance"),
    spark_sum((col("fare_amount") < 0).cast("int")).alias("negative_fare"),
    spark_sum((col("trip_duration_minutes") <= 0).cast("int")).alias("non_positive_duration")
).show()

+---------------------+-------------+---------------------+
|non_positive_distance|negative_fare|non_positive_duration|
+---------------------+-------------+---------------------+
|               773457|       381650|                15569|
+---------------------+-------------+---------------------+



In [35]:
from pyspark.sql.functions import col

invalid_df = trips_joined_df.filter(
    (col("trip_distance") <= 0) |
    (col("fare_amount") < 0) |
    (col("trip_duration_minutes") <= 0)
)

print("Records failing at least one validity check:", invalid_df.count())

Records failing at least one validity check: 1117974


In [36]:
trips_joined_df.filter(
    (col("pickup_date") < "2023-01-01") |
    (col("pickup_date") > "2023-12-31")
).select(
    "pickup_date"
).groupBy(
    "pickup_date"
).count().orderBy(
    "pickup_date"
).show(50)

+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2001-01-01|    6|
| 2002-12-31|   11|
| 2003-01-01|    6|
| 2008-12-31|   23|
| 2009-01-01|   15|
| 2014-11-19|    1|
| 2022-10-24|    4|
| 2022-10-25|    7|
| 2022-12-31|   25|
| 2024-01-01|    2|
| 2024-01-03|    4|
+-----------+-----+



In [37]:
valid_trips_df = trips_joined_df.filter(
    (col("pickup_date") >= "2023-01-01") &
    (col("pickup_date") <= "2023-12-31") &
    (col("trip_distance") > 0) &
    (col("fare_amount") >= 0) &
    (col("trip_duration_minutes") > 0)
)

In [38]:
total_records = trips_joined_df.count()
valid_records = valid_trips_df.count()

print("Total records:", total_records)
print("Valid records:", valid_records)
print("Records excluded:", total_records - valid_records)

Total records: 38310226
Valid records: 37192160
Records excluded: 1118066


### 2.6 Analytical Projection and Route Aggregation

In [39]:
analysis_df = valid_trips_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "pickup_zone",
    "dropoff_zone",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount"
)

#### Route representation

The approved proposal identified a derived `route_id` constructed from the pickup and drop off location IDs. In the implementation, the route is represented directly using the combination of `PULocationID` and `DOLocationID` rather than creating an additional encoded `route_id` column.

This preserves the same route level grouping required by the business query while keeping the original pickup and drop off location identifiers available for aggregation, validation and interpretation.

In [40]:
from pyspark.sql.functions import col, count, sum as spark_sum

MIN_TRIPS = 100

route_agg_df = (
    analysis_df
    .groupBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "pickup_zone",
        "dropoff_zone"
    )
    .agg(
        count("*").alias("trip_count"),
        spark_sum("trip_distance").alias("total_distance"),
        spark_sum("trip_duration_minutes").alias("total_duration_minutes"),
        spark_sum("fare_amount").alias("total_fare")
    )
    .filter(col("trip_count") >= MIN_TRIPS)
    .withColumn(
        "route_speed_mph",
        col("total_distance") /
        (col("total_duration_minutes") / 60.0)
    )
    .withColumn(
        "fare_per_occupied_minute",
        col("total_fare") /
        col("total_duration_minutes")
    )
)

In [41]:
route_agg_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    spark_round("route_speed_mph", 2).alias("speed_mph")
).limit(5).show(truncate=22)

+--------------+-----------+-----------+----------------------+-------------------+----------+---------+
|pickup_borough|day_of_week|pickup_hour|           pickup_zone|       dropoff_zone|trip_count|speed_mph|
+--------------+-----------+-----------+----------------------+-------------------+----------+---------+
|        Queens|          1|          0|           JFK Airport|Crown Heights North|       125|     22.5|
|     Manhattan|          1|          0|Sutton Place/Turtle...|UN/Turtle Bay South|       112|    11.68|
|     Manhattan|          1|          0| Upper West Side South|     Yorkville West|       221|    11.54|
|     Manhattan|          1|          0|Meatpacking/West Vi...|           Flatiron|       210|     9.73|
|     Manhattan|          1|          0|              Flatiron|           Flatiron|       103|     9.38|
+--------------+-----------+-----------+----------------------+-------------------+----------+---------+



### 2.7 Borough and Time Peer Benchmark

In [42]:
benchmark_df = (
    analysis_df
    .groupBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour"
    )
    .agg(
        spark_sum("trip_distance").alias("benchmark_total_distance"),
        spark_sum("trip_duration_minutes").alias("benchmark_total_duration"),
        spark_sum("fare_amount").alias("benchmark_total_fare")
    )
    .withColumn(
        "benchmark_speed_mph",
        col("benchmark_total_distance") /
        (col("benchmark_total_duration") / 60.0)
    )
    .withColumn(
        "benchmark_fare_per_minute",
        col("benchmark_total_fare") /
        col("benchmark_total_duration")
    )
)

In [43]:
benchmark_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    spark_round("benchmark_speed_mph", 2).alias("benchmark_speed"),
    spark_round("benchmark_fare_per_minute", 2).alias("benchmark_fare_min")
).show(10, truncate=False)

+--------------+-----------+-----------+---------------+------------------+
|pickup_borough|day_of_week|pickup_hour|benchmark_speed|benchmark_fare_min|
+--------------+-----------+-----------+---------------+------------------+
|N/A           |2          |3          |21.12          |12.61             |
|Unknown       |7          |21         |12.06          |1.08              |
|N/A           |5          |15         |12.74          |1.16              |
|Bronx         |1          |10         |14.83          |1.06              |
|Manhattan     |1          |15         |13.34          |1.05              |
|N/A           |3          |13         |15.02          |1.52              |
|N/A           |5          |13         |15.83          |2.05              |
|Staten Island |1          |10         |25.34          |1.21              |
|Bronx         |4          |2          |23.62          |2.54              |
|Staten Island |7          |14         |19.21          |1.04              |
+-----------

### 2.8 Route Comparison and Window Ranking

In [44]:
route_comparison_df = (
    route_agg_df
    .join(
        benchmark_df,
        on=[
            "pickup_borough",
            "day_of_week",
            "pickup_hour"
        ],
        how="inner"
    )
    .withColumn(
        "speed_deficit_percentage",
        (
            (
                col("benchmark_speed_mph") -
                col("route_speed_mph")
            )
            / col("benchmark_speed_mph")
        ) * 100
    )
)

In [45]:
route_comparison_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    spark_round("route_speed_mph", 2).alias("route_speed"),
    spark_round("benchmark_speed_mph", 2).alias("benchmark_speed"),
    spark_round("speed_deficit_percentage", 2).alias("deficit_pct")
).limit(5).show(truncate=20)

+--------------+-----------+-----------+--------------------+----------------+----------+-----------+---------------+-----------+
|pickup_borough|day_of_week|pickup_hour|         pickup_zone|    dropoff_zone|trip_count|route_speed|benchmark_speed|deficit_pct|
+--------------+-----------+-----------+--------------------+----------------+----------+-----------+---------------+-----------+
|     Manhattan|          1|         15|      Midtown Center|  Yorkville East|       153|        9.6|          13.34|      28.05|
|     Manhattan|          1|         15|Greenwich Village...|        Flatiron|       138|       9.62|          13.34|      27.85|
|     Manhattan|          1|         15|        East Chelsea|Garment District|       180|       5.29|          13.34|       60.3|
|     Manhattan|          1|         15|    Garment District|     Murray Hill|       121|       1.63|          13.34|      87.77|
|     Manhattan|          1|         15|Times Sq/Theatre ...|Garment District|       248| 

In [46]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

route_window = (
    Window
    .partitionBy(
        "pickup_borough",
        "day_of_week",
        "pickup_hour"
    )
    .orderBy(
        col("speed_deficit_percentage").desc()
    )
)

ranked_routes_df = (
    route_comparison_df
    .withColumn(
        "route_rank",
        row_number().over(route_window)
    )
    .filter(
        col("route_rank") <= 3
    )
)

In [47]:
ranked_routes_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "route_rank",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    spark_round("route_speed_mph", 2).alias("route_speed"),
    spark_round("benchmark_speed_mph", 2).alias("benchmark_speed"),
    spark_round("speed_deficit_percentage", 2).alias("deficit_pct")
).orderBy("route_rank").show(truncate=20)

+----------+----------------+----------------+----------+-----------+---------------+-----------+
|route_rank|     pickup_zone|    dropoff_zone|trip_count|route_speed|benchmark_speed|deficit_pct|
+----------+----------------+----------------+----------+-----------+---------------+-----------+
|         1|   Midtown South|   Midtown South|       119|       1.45|          13.34|      89.13|
|         2|Garment District|     Murray Hill|       121|       1.63|          13.34|      87.77|
|         3|    Clinton East|Garment District|       119|       2.16|          13.34|      83.82|
+----------+----------------+----------------+----------+-----------+---------------+-----------+



In [48]:
ranked_routes_df = (
    ranked_routes_df
    .withColumn(
        "lower_fare_than_benchmark",
        col("fare_per_occupied_minute") <
        col("benchmark_fare_per_minute")
    )
)

In [49]:
ranked_routes_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "route_rank",
    "pickup_zone",
    "dropoff_zone",
    spark_round("fare_per_occupied_minute", 2).alias("route_fare_min"),
    spark_round("benchmark_fare_per_minute", 2).alias("benchmark_fare_min"),
    col("lower_fare_than_benchmark").alias("lower_than_benchmark")
).orderBy("route_rank").show(truncate=20)

+----------+----------------+----------------+--------------+------------------+--------------------+
|route_rank|     pickup_zone|    dropoff_zone|route_fare_min|benchmark_fare_min|lower_than_benchmark|
+----------+----------------+----------------+--------------+------------------+--------------------+
|         1|   Midtown South|   Midtown South|           0.5|              1.05|                true|
|         2|Garment District|     Murray Hill|          0.32|              1.05|                true|
|         3|    Clinton East|Garment District|          0.52|              1.05|                true|
+----------+----------------+----------------+--------------+------------------+--------------------+



### 2.9 DataFrame Optimisation Discussion

The implementation removes records outside 2023, non positive trip distances, negative fares and non positive trip durations before the analytical aggregations. This reduces the number of records carried into the route and peer benchmark calculations.

The `analysis_df` projection retains only the columns required by the business query. The physical plan also shows that Spark applies filtering close to the Parquet scans before the Taxi Zone Lookup joins. Supported predicates are shown in the Parquet `PushedFilters`, while the remaining validity conditions are applied by the `Filter` operator before the joins. This reduces the amount of trip data carried into later operations.

The physical plan shows that the small Taxi Zone Lookup tables are joined using `BroadcastHashJoin`, avoiding repartitioning the much larger trip dataset for these joins.

No intermediate DataFrame was explicitly cached. The Docker environment has limited memory and the valid trip dataset contains more than 37 million records, so the final implementation does not retain this dataset in memory.

## 3. Spark SQL Implementation

### 3.1 Temporary View and SQL Query

In [50]:
analysis_df.createOrReplaceTempView("valid_trips")

In [51]:
sql_result_df = spark.sql("""
WITH route_agg AS (
    SELECT
        pickup_borough,
        day_of_week,
        pickup_hour,
        PULocationID,
        DOLocationID,
        pickup_zone,
        dropoff_zone,
        COUNT(*) AS trip_count,
        SUM(trip_distance) AS total_distance,
        SUM(trip_duration_minutes) AS total_duration_minutes,
        SUM(fare_amount) AS total_fare
    FROM valid_trips
    GROUP BY
        pickup_borough,
        day_of_week,
        pickup_hour,
        PULocationID,
        DOLocationID,
        pickup_zone,
        dropoff_zone
    HAVING COUNT(*) >= 100
),

route_metrics AS (
    SELECT
        *,
        total_distance / (total_duration_minutes / 60.0)
            AS route_speed_mph,
        total_fare / total_duration_minutes
            AS fare_per_occupied_minute
    FROM route_agg
),

benchmark AS (
    SELECT
        pickup_borough,
        day_of_week,
        pickup_hour,
        SUM(trip_distance) / (SUM(trip_duration_minutes) / 60.0)
            AS benchmark_speed_mph,
        SUM(fare_amount) / SUM(trip_duration_minutes)
            AS benchmark_fare_per_minute
    FROM valid_trips
    GROUP BY
        pickup_borough,
        day_of_week,
        pickup_hour
),

comparison AS (
    SELECT
        r.*,
        b.benchmark_speed_mph,
        b.benchmark_fare_per_minute,
        (
            (b.benchmark_speed_mph - r.route_speed_mph)
            / b.benchmark_speed_mph
        ) * 100 AS speed_deficit_percentage
    FROM route_metrics r
    INNER JOIN benchmark b
        ON r.pickup_borough = b.pickup_borough
        AND r.day_of_week = b.day_of_week
        AND r.pickup_hour = b.pickup_hour
),

ranked AS (
    SELECT
        *,
        fare_per_occupied_minute < benchmark_fare_per_minute
            AS lower_fare_than_benchmark,
        ROW_NUMBER() OVER (
            PARTITION BY
                pickup_borough,
                day_of_week,
                pickup_hour
            ORDER BY speed_deficit_percentage DESC
        ) AS route_rank
    FROM comparison
)

SELECT *
FROM ranked
WHERE route_rank <= 3
""")

### 3.2 Spark SQL Results

In [52]:
sql_result_df.filter(
    (col("pickup_borough") == "Manhattan") &
    (col("day_of_week") == 1) &
    (col("pickup_hour") == 15)
).select(
    "route_rank",
    "pickup_zone",
    "dropoff_zone",
    "trip_count",
    spark_round("speed_deficit_percentage", 2).alias("deficit_pct"),
    col("lower_fare_than_benchmark").alias("lower_than_benchmark")
).orderBy("route_rank").show(truncate=20)

+----------+----------------+----------------+----------+-----------+--------------------+
|route_rank|     pickup_zone|    dropoff_zone|trip_count|deficit_pct|lower_than_benchmark|
+----------+----------------+----------------+----------+-----------+--------------------+
|         1|   Midtown South|   Midtown South|       119|      89.13|                true|
|         2|Garment District|     Murray Hill|       121|      87.77|                true|
|         3|    Clinton East|Garment District|       119|      83.82|                true|
+----------+----------------+----------------+----------+-----------+--------------------+



### 3.3 DataFrame and Spark SQL Comparison

The DataFrame implementation expresses the analytical query through chained transformations, while the Spark SQL implementation separates the route aggregation, peer benchmark, comparison and ranking into Common Table Expressions. The SQL structure makes each stage of the query explicit, while the DataFrame approach integrates naturally with the earlier Spark preprocessing and transformation steps. Both implementations express the same analytical logic, but use different interfaces for constructing the query.

## 4. Result Validation

### 4.1 Full Result Comparison

In [53]:
df_validation = ranked_routes_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "trip_count",
    "route_rank"
)

sql_validation = sql_result_df.select(
    "pickup_borough",
    "day_of_week",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "trip_count",
    "route_rank"
)

print("DataFrame result rows:", df_validation.count())
print("SQL result rows:", sql_validation.count())

print(
    "DataFrame rows not in SQL:",
    df_validation.exceptAll(sql_validation).count()
)

print(
    "SQL rows not in DataFrame:",
    sql_validation.exceptAll(df_validation).count()
)

DataFrame result rows: 1093
SQL result rows: 1093
DataFrame rows not in SQL: 0
SQL rows not in DataFrame: 0


### 4.2 Validation Discussion

The DataFrame and Spark SQL implementations each returned 1,093 result rows. The `exceptAll()` comparisons returned zero unmatched rows in both directions, confirming that the two implementations produced equivalent analytical results for the selected validation fields.

# Part B: System Perspective and Performance Analysis

## 1. Partitioning Strategy

### 1.1 Partition Column and Number of Partitions

The partitioning experiment uses `pickup_epoch_seconds`, the high cardinality numerical field identified in the approved proposal. This field represents pickup time as Unix epoch seconds and provides a meaningful chronological ordering for comparing hash and range partitioning.

The number of partitions was set to twice `sc.defaultParallelism`. Spark runs with `local[2]`, giving a default parallelism of 2, so both strategies use 4 partitions. This provides multiple tasks per available core while keeping the same partition count for both methods so their distributions can be compared consistently.

In [54]:
partition_source_df = valid_trips_df.select(
    "pickup_epoch_seconds"
)

In [55]:
num_partitions = sc.defaultParallelism * 2

print("Number of partitions:", num_partitions)

Number of partitions: 4


### 1.2 Hash Partitioning

In [56]:
hash_partitioned_df = (
    partition_source_df
    .repartition(
        num_partitions,
        "pickup_epoch_seconds"
    )
)

In [57]:
from pyspark.sql.functions import spark_partition_id

hash_partition_counts = (
    hash_partitioned_df
    .withColumn(
        "partition_id",
        spark_partition_id()
    )
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

hash_partition_counts.show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|9296151|
|           1|9300112|
|           2|9296479|
|           3|9299418|
+------------+-------+



### 1.3 Range Partitioning

In [58]:
range_partitioned_df = (
    partition_source_df
    .repartitionByRange(
        num_partitions,
        "pickup_epoch_seconds"
    )
)

In [59]:
range_partition_counts = (
    range_partitioned_df
    .withColumn(
        "partition_id",
        spark_partition_id()
    )
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

range_partition_counts.show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|9270087|
|           1|9319987|
|           2|9278490|
|           3|9323596|
+------------+-------+



### 1.4 Comparative Analysis

Both partitioning strategies distributed the 37,192,160 valid trip records across four partitions. Hash partitioning produced 9,296,151, 9,300,112, 9,296,479 and 9,299,418 records. The difference between the largest and smallest partitions was only 3,961 records, showing a very even distribution with no substantial evidence of skew.

Range partitioning produced 9,270,087, 9,319,987, 9,278,490 and 9,323,596 records. The difference between the largest and smallest partitions was 53,509 records. The range partitions were therefore also relatively balanced, although the variation was greater than for hash partitioning.

Hash partitioning distributes values according to the hash of `pickup_epoch_seconds`, which produced the most even partition sizes in this experiment. Range partitioning instead assigns contiguous timestamp ranges to partitions and therefore preserves the chronological ordering of the selected field.

For this experiment, hash partitioning provided the more balanced workload distribution. Range partitioning also avoided severe skew while retaining temporal locality. The results demonstrate that the two strategies distribute the same records differently while using the same number of partitions.

## 2. Execution Time Benchmarking

### 2.1 DataFrame Benchmark

In [60]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def build_ranked_routes():

    route_agg = (
        analysis_df
        .groupBy(
            "pickup_borough",
            "day_of_week",
            "pickup_hour",
            "PULocationID",
            "DOLocationID",
            "pickup_zone",
            "dropoff_zone"
        )
        .agg(
            F.count("*").alias("trip_count"),
            F.sum("trip_distance").alias("total_distance"),
            F.sum("trip_duration_minutes").alias("total_duration_minutes"),
            F.sum("fare_amount").alias("total_fare")
        )
        .filter(F.col("trip_count") >= 100)
        .withColumn(
            "route_speed_mph",
            F.col("total_distance") /
            (F.col("total_duration_minutes") / 60.0)
        )
        .withColumn(
            "fare_per_occupied_minute",
            F.col("total_fare") /
            F.col("total_duration_minutes")
        )
    )

    benchmark = (
        analysis_df
        .groupBy(
            "pickup_borough",
            "day_of_week",
            "pickup_hour"
        )
        .agg(
            F.sum("trip_distance").alias("benchmark_total_distance"),
            F.sum("trip_duration_minutes").alias("benchmark_total_duration"),
            F.sum("fare_amount").alias("benchmark_total_fare")
        )
        .withColumn(
            "benchmark_speed_mph",
            F.col("benchmark_total_distance") /
            (F.col("benchmark_total_duration") / 60.0)
        )
        .withColumn(
            "benchmark_fare_per_minute",
            F.col("benchmark_total_fare") /
            F.col("benchmark_total_duration")
        )
    )

    comparison = (
        route_agg
        .join(
            benchmark,
            ["pickup_borough", "day_of_week", "pickup_hour"],
            "inner"
        )
        .withColumn(
            "speed_deficit_percentage",
            (
                (
                    F.col("benchmark_speed_mph") -
                    F.col("route_speed_mph")
                )
                / F.col("benchmark_speed_mph")
            ) * 100
        )
    )

    route_window = (
        Window
        .partitionBy(
            "pickup_borough",
            "day_of_week",
            "pickup_hour"
        )
        .orderBy(
            F.col("speed_deficit_percentage").desc()
        )
    )

    return (
        comparison
        .withColumn(
            "route_rank",
            F.row_number().over(route_window)
        )
        .filter(F.col("route_rank") <= 3)
        .withColumn(
            "lower_fare_than_benchmark",
            F.col("fare_per_occupied_minute") <
            F.col("benchmark_fare_per_minute")
        )
    )

In [61]:
%%time

run_times = []

for i in range(3):
    spark.catalog.clearCache()

    start_time = time.perf_counter()

    benchmark_df = build_ranked_routes()
    benchmark_output = benchmark_df.collect()

    elapsed_time = time.perf_counter() - start_time
    run_times.append(elapsed_time)

    print(
        f"Run {i + 1}: {elapsed_time:.2f} seconds, "
        f"Rows returned: {len(benchmark_output)}"
    )

median_time = statistics.median(run_times)

print("\nRun times:", [round(t, 2) for t in run_times])
print(f"Median runtime: {median_time:.2f} seconds")

Run 1: 142.40 seconds, Rows returned: 1093
Run 2: 130.39 seconds, Rows returned: 1093
Run 3: 135.83 seconds, Rows returned: 1093

Run times: [142.4, 130.39, 135.83]
Median runtime: 135.83 seconds
CPU times: user 1.35 s, sys: 408 ms, total: 1.75 s
Wall time: 6min 48s


The complete DataFrame business query was executed three times in the Docker environment using Spark local mode with 2 processing cores and 2 GB driver memory. Each run rebuilt the query and used `collect()` to trigger execution of the full Spark lineage. All runs returned 1,093 rows.

The median runtime was calculated programmatically using Python's `statistics.median()` function. Individual execution times may vary between runs because of system resource availability and runtime conditions, so the reported median represents the observed performance for this execution environment.

### 2.2 Spark SQL Benchmark

In [62]:
def build_sql_result():
    return spark.sql("""
    WITH route_agg AS (
        SELECT
            pickup_borough,
            day_of_week,
            pickup_hour,
            PULocationID,
            DOLocationID,
            pickup_zone,
            dropoff_zone,
            COUNT(*) AS trip_count,
            SUM(trip_distance) AS total_distance,
            SUM(trip_duration_minutes) AS total_duration_minutes,
            SUM(fare_amount) AS total_fare
        FROM valid_trips
        GROUP BY
            pickup_borough,
            day_of_week,
            pickup_hour,
            PULocationID,
            DOLocationID,
            pickup_zone,
            dropoff_zone
        HAVING COUNT(*) >= 100
    ),

    route_metrics AS (
        SELECT
            *,
            total_distance / (total_duration_minutes / 60.0)
                AS route_speed_mph,
            total_fare / total_duration_minutes
                AS fare_per_occupied_minute
        FROM route_agg
    ),

    benchmark AS (
        SELECT
            pickup_borough,
            day_of_week,
            pickup_hour,
            SUM(trip_distance) / (SUM(trip_duration_minutes) / 60.0)
                AS benchmark_speed_mph,
            SUM(fare_amount) / SUM(trip_duration_minutes)
                AS benchmark_fare_per_minute
        FROM valid_trips
        GROUP BY
            pickup_borough,
            day_of_week,
            pickup_hour
    ),

    comparison AS (
        SELECT
            r.*,
            b.benchmark_speed_mph,
            b.benchmark_fare_per_minute,
            (
                (b.benchmark_speed_mph - r.route_speed_mph)
                / b.benchmark_speed_mph
            ) * 100 AS speed_deficit_percentage
        FROM route_metrics r
        INNER JOIN benchmark b
            ON r.pickup_borough = b.pickup_borough
            AND r.day_of_week = b.day_of_week
            AND r.pickup_hour = b.pickup_hour
    ),

    ranked AS (
        SELECT
            *,
            fare_per_occupied_minute < benchmark_fare_per_minute
                AS lower_fare_than_benchmark,
            ROW_NUMBER() OVER (
                PARTITION BY
                    pickup_borough,
                    day_of_week,
                    pickup_hour
                ORDER BY speed_deficit_percentage DESC
            ) AS route_rank
        FROM comparison
    )

    SELECT *
    FROM ranked
    WHERE route_rank <= 3
    """)

In [63]:
%%time

sql_run_times = []

for i in range(3):
    spark.catalog.clearCache()

    start_time = time.perf_counter()

    sql_benchmark_df = build_sql_result()
    sql_benchmark_output = sql_benchmark_df.collect()

    elapsed_time = time.perf_counter() - start_time
    sql_run_times.append(elapsed_time)

    print(
        f"Run {i + 1}: {elapsed_time:.2f} seconds, "
        f"Rows returned: {len(sql_benchmark_output)}"
    )

sql_median_time = statistics.median(sql_run_times)

print("\nSQL run times:", [round(t, 2) for t in sql_run_times])
print(f"SQL median runtime: {sql_median_time:.2f} seconds")

Run 1: 172.18 seconds, Rows returned: 1093
Run 2: 241.02 seconds, Rows returned: 1093
Run 3: 178.78 seconds, Rows returned: 1093

SQL run times: [172.18, 241.02, 178.78]
SQL median runtime: 178.78 seconds
CPU times: user 2.73 s, sys: 240 ms, total: 2.97 s
Wall time: 9min 51s


The complete Spark SQL business query was executed three times in the same Docker environment using Spark local mode with 2 processing cores and 2 GB driver memory. Each run rebuilt the SQL query and used `collect()` to trigger execution of the full query. All three runs returned 1,093 rows.

The recorded SQL runtimes were 172.18 seconds, 241.02 seconds and 178.78 seconds. The median runtime was calculated programmatically using Python's `statistics.median()` function and was 178.78 seconds. Individual runtimes varied between runs, so the median is used as the representative execution time for this environment.

### 2.3 Runtime Comparison

In [64]:
performance_comparison = [
    ("DataFrame API", median_time),
    ("Spark SQL", sql_median_time)
]

performance_df = spark.createDataFrame(
    performance_comparison,
    ["Implementation", "Median Runtime Seconds"]
)

performance_df.select(
    "Implementation",
    spark_round("Median Runtime Seconds", 2).alias("Median Runtime Seconds")
).show(truncate=False)

+--------------+----------------------+
|Implementation|Median Runtime Seconds|
+--------------+----------------------+
|DataFrame API |135.83                |
|Spark SQL     |178.78                |
+--------------+----------------------+



### 2.4 Performance Discussion

The DataFrame API produced a median runtime of 135.83 seconds, while Spark SQL produced a median runtime of 178.78 seconds. In this execution environment, the Spark SQL implementation therefore took approximately 42.95 seconds longer than the DataFrame implementation.

Both implementations express the same analytical logic and are processed by Spark's Catalyst optimiser before execution. Both require the same major operations, including route aggregation, peer benchmark aggregation, joining the aggregated results and window ranking. The observed timing difference should therefore be interpreted as the result of these runs in the local Docker environment rather than evidence that one interface is always faster.

The physical plan contains several shuffle operations associated with aggregation and joining. These operations redistribute records between partitions and introduce additional stages. Runtime can also vary between runs because of file reading, JVM execution and the limited resources available in the local environment.

For the recorded runs, the DataFrame implementation had the lower median runtime, while both implementations returned the same 1,093 analytical results.

## 3. Execution Plan Interpretation

### 3.1 Extended Execution Plan

In [65]:
ranked_routes_df.explain(extended=True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(lower_fare_than_benchmark, '`<`('fare_per_occupied_minute, 'benchmark_fare_per_minute), None)]
+- Filter (route_rank#2428 <= 3)
   +- Project [pickup_borough#1087, day_of_week#961, pickup_hour#960, PULocationID#381L, DOLocationID#382L, pickup_zone#1088, dropoff_zone#1092, trip_count#1583L, total_distance#1584, total_duration_minutes#1585, total_fare#1586, route_speed_mph#1601, fare_per_occupied_minute#1602, benchmark_total_distance#1687, benchmark_total_duration#1688, benchmark_total_fare#1689, benchmark_speed_mph#1703, benchmark_fare_per_minute#1704, speed_deficit_percentage#2276, route_rank#2428]
      +- Project [pickup_borough#1087, day_of_week#961, pickup_hour#960, PULocationID#381L, DOLocationID#382L, pickup_zone#1088, dropoff_zone#1092, trip_count#1583L, total_distance#1584, total_duration_minutes#1585, total_fare#1586, route_speed_mph#1601, fare_per_occupied_minute#1602, benchmark_total_distance#1687, benchmark_total

### 3.2 Physical Plan Identification

The physical plan contains `Exchange hashpartitioning` operations for the route aggregation, the borough and time benchmark aggregation, and the join between the two aggregated results. The route aggregation uses a partial `HashAggregate`, followed by an exchange and a final `HashAggregate`. The route and benchmark results are also repartitioned by pickup borough, day of week and pickup hour before the `SortMergeJoin`. The Taxi Zone Lookup tables are joined using `BroadcastHashJoin`.

The following shuffle points are identified from the physical plan:

`HashAggregate` → `Exchange hashpartitioning(pickup_borough, day_of_week, pickup_hour, PULocationID, DOLocationID, pickup_zone, dropoff_zone, 200)`

This exchange occurs between the partial and final route aggregation.

`HashAggregate` → `Exchange hashpartitioning(pickup_borough, day_of_week, pickup_hour, 200)`

This exchange occurs between the partial and final peer benchmark aggregation.

`Project` → `Exchange hashpartitioning(pickup_borough, day_of_week, pickup_hour, 200)` → `Sort` → `SortMergeJoin`

This exchange repartitions the route results using the join keys before the route and peer benchmark results are joined.

### 3.3 Execution Plan Discussion

The physical plan shows several `Exchange hashpartitioning` operations where Spark redistributes records between partitions. During the route aggregation, a partial `HashAggregate` is followed by an exchange using the route and time grouping columns before the final aggregation. This shuffle is required because records belonging to the same route and time group may initially be located in different partitions.

Another exchange occurs before the `SortMergeJoin`. The route results and peer benchmark results are repartitioned using pickup borough, day of week and pickup hour so matching records can be brought together. The peer benchmark aggregation also requires an exchange between its partial and final aggregation stages.

In a distributed environment these shuffles increase network I/O. In the current local environment they still require shuffle data movement and create additional stage boundaries, which can increase task scheduling and execution overhead.

The Taxi Zone Lookup joins avoid a larger shuffle because Spark uses `BroadcastHashJoin` for the small lookup tables. Filtering and projecting required columns before later aggregation and shuffle operations also reduces the amount of data that must be processed.

## 4. Spark Web UI DAG

### 4.1 DAG Screenshot
 
<img src="./dag.png" alt="Spark Web UI DAG" width="100%">

### 4.2 DAG Discussion

The Spark Web UI DAG shows multiple execution regions separated by shuffle processing. The visible `Exchange` represents redistribution of records between partitions, while the following `AQEShuffleRead` shows Spark reading the resulting shuffle data before continuing with the downstream aggregation.

After the shuffle, the DAG shows a `HashAggregate` followed by `Filter` and `Project` operations. It also shows a `BroadcastHashJoin`, consistent with the physical execution plan where the small Taxi Zone Lookup table is broadcast rather than requiring the larger trip dataset to be repartitioned. A later `Sort` operation is also visible as part of the analytical processing.

No obvious severe skew can be concluded from the DAG alone because it primarily shows the execution structure rather than the full distribution of records between tasks. The separate partitioning experiment provides the clearer evidence and showed relatively balanced partition sizes for both strategies.

The main opportunities for reducing shuffle work remain limiting the columns and records carried into later operations and using broadcast joins for the small lookup data.